# TaskB: Matching the Job-Title with the Relevant Skills for the Development Set

In this notebook, we prepare the shared-track Task-B dataset 
- for developing the job-title to Skills matching model and
- for evaluating the predicted matched skills using evaluation script [task's evaluation script](https://github.com/TalentCLEF/talentclef25_evaluation_script).

- Additionally, the provided format is also compatible with the testset.

## Import Required Libraries

In [4]:
import os
import sys
import json

In [5]:
import pandas as pd
import numpy as np
import subprocess
import ast

In [6]:
from sentence_transformers import SentenceTransformer, util
from codecarbon import EmissionsTracker

## Matching the JobTitle for Validation data 
- Preparing data (Task B)

### Setting up data path

In [7]:
root_data_path="taskB"

In [8]:
#Training data path
taskB_training_path = os.path.join(root_data_path, "training")

#validation data path
validation_data_path=os.path.join(root_data_path, "validation")

#testing data path
test_data_path=os.path.join(root_data_path, "test")

#model path
models_path = "models"

#results path
results_path = "results"

#evaluation result path
evaluations_path = "evaluations"

### Training data loading

#### Training data dictionary

The training data for Task B is shared in 3 files:
- `job2skill.tsv`: This file has been curated to include the most representative skills for each job title in <a href="https://esco.ec.europa.eu/en">ESCO</a>. This file contains three columns:
  - `job_id`: ESCO identifier for the job position.
  - `skill_id`: ESCO identifier for the skill.
  - `rel_type`: Indicator specifying whether the skill_id is essential or optional for a specific job_id. It can have the value “essential” or “optional.”

- `jobid2terms.json`: This JSON file contains job_id identifiers used in the training set for Task B as keys, and a list of valid lexical variants for each identifier as values.

- `skillid2terms.json`: This JSON file contains skill_id identifiers as keys, and a list of valid lexical variants for each identifier as values.


#### Reading Job2Skill data file

In [9]:
# Read job2skill file
job2skill = pd.read_csv(os.path.join(taskB_training_path, 'job2skill.tsv'),
                        sep="\t",
                        names=["job_id","skill_id","rel_type"])

print ("No. of job_id and skill_id triple: {}".format(job2skill.shape[0]))
job2skill.head()

No. of job_id and skill_id triple: 114699


,job_id,skill_id,rel_type
0,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/93a68dcb-3dc6...,essential
1,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/05bc7677-5a64...,essential
2,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/860be36a-d19b...,essential
3,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/fed5b267-73fa...,essential
4,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/f64fe2c2-d090...,essential


#### Reading Job2Terms file

In [10]:
# Read json files
jobid2terms_path = os.path.join(taskB_training_path, "jobid2terms.json")
print (jobid2terms_path)

taskB/training/jobid2terms.json


In [11]:
with open(os.path.join(jobid2terms_path), 'r') as file:
    jobid2terms = json.load(file)

In [12]:
print ("Total job ids: {}".format(len(jobid2terms)))

Total job ids: 3039


#### Reading Skilled2Terms

In [13]:
# Read json files
skillid2terms_path = os.path.join(taskB_training_path, "skillid2terms.json")
print (skillid2terms_path)

taskB/training/skillid2terms.json


In [14]:
with open(os.path.join(skillid2terms_path), 'r') as file:
    skillid2terms = json.load(file)

In [15]:
print ("Total skill ids: {}".format(len(skillid2terms)))

Total skill ids: 13939


#### Play with Job-id, Skill-id, Job-id-terms, and Skill-id-terms. 

##### Expanding Job2Skill with Terms Mapping

In [16]:
job2skill["job_terms"] = job2skill["job_id"].map(jobid2terms)
job2skill["skill_terms"] = job2skill["skill_id"].map(skillid2terms)

In [17]:
job2skill.describe()

,job_id,skill_id,rel_type,job_terms,skill_terms
count,114699,114699,114699,114699,114699
unique,3011,13224,2,3011,13224
top,http://data.europa.eu/esco/occupation/8c0f59c9...,http://data.europa.eu/esco/skill/03b9b491-fc9b...,essential,"[microelectronics engineer, microelectronics t...","[create solutions to problems, create solution..."
freq,69,328,62480,69,328


**Let's see the skills for a specific job_id.**



In [18]:
job_id_to_search = job2skill.job_id.to_list()[100]
job_data = job2skill[job2skill.job_id == job_id_to_search]

**The job titles of the selected job_id are:**

In [19]:
job_data.job_terms.iloc[0]

['air traffic safety technician',
 'air traffic safety electronics hardware specialist',
 'air traffic safety software specialist',
 'air traffic safety engineer',
 'air traffic safety hardware specialist',
 'air traffic safety electronics software specialist',
 'air traffic safety electronics engineer',
 'air traffic safety electronics technician']

**For that job_id we fin 9 essential skills and 1 optional skill:**

In [20]:
job_data.rel_type.value_counts()

rel_type
optional     31
essential    24
Name: count, dtype: int64

The list of skills related to the job_id are:

In [21]:
job_data.skill_terms.to_list()[0]

['common aviation safety regulations',
 'common safety regulations in aviation',
 'standard regulations in aviation safety',
 'prevailing safety regulations in civil aviation',
 'regulations governing safety in international civil aviation',
 'common safety legislation in aviation',
 'common civil aviation safety regulations',
 'prevailing aviation safety directives',
 'common aviation safety legislation',
 'common international civil aviation safety regulations',
 'standard aviation safety regulations']

In [22]:
job2skill.job_id.to_list()[100]

'http://data.europa.eu/esco/occupation/0022f466-426c-41a4-ac96-a235c945cf97'

### **Validation data loading**

#### **Validation data path setup**

In [23]:
validation_queries_path = os.path.join(validation_data_path, "queries")
validation_corpus_elements_path = os.path.join(validation_data_path, "corpus_elements")

#### **Load queries and corpus elements**

In [24]:
validation_queries = pd.read_csv(validation_queries_path, sep="\t")
validation_corpus_elements = pd.read_csv(validation_corpus_elements_path, sep="\t")

In [25]:
print ("Number of queries and copurs elements in validation dataset: {} and {}".format(len(validation_queries), len(validation_corpus_elements)))

Number of queries and copurs elements in validation dataset: 304 and 1439


In [26]:
print ("validation query samples:")
validation_queries.head()

validation query samples:


,q_id,jobtitle
0,dev_qb_jt_1,corporate governance analyst
1,dev_qb_jt_2,logistics business analyst
2,dev_qb_jt_3,operations planning analyst
3,dev_qb_jt_4,real estate data analyst
4,dev_qb_jt_5,security analyst ii


In [27]:
validation_queries.jobtitle[0]

'corporate governance analyst'

In [28]:
print ("validation corpus elements samples:")
validation_corpus_elements.head()

validation corpus elements samples:


,c_id,esco_uri,skill_aliases
0,dev_cb_sk_1,http://data.europa.eu/esco/skill/1c460d2d-90c6...,"['pricing plans', 'price strategies', 'pricing..."
1,dev_cb_sk_2,http://data.europa.eu/esco/skill/301a6581-e983...,"['putting out fires', 'put out fires', 'exting..."
2,dev_cb_sk_3,http://data.europa.eu/esco/skill/a4881e54-6055...,"['online assessment', 'analysis of web strateg..."
3,dev_cb_sk_4,http://data.europa.eu/esco/skill/efda73b4-5212...,"['keeping up with trends', 'keep pace with tre..."
4,dev_cb_sk_5,http://data.europa.eu/esco/skill/22a173f5-868c...,"['prepare tax returns form', 'complete tax ret..."


In [29]:
validation_corpus_elements.skill_aliases[0]

"['pricing plans', 'price strategies', 'pricing tactics', 'pricing strategies', 'pricing strategy']"

### **Matching JobTitle with Skills**

- *Idea 1: Adhoc (no machine learning learning)*
    - Preprocessing: Aggregate the skill aliases into one long text
    - Task: Match the job title text with the aggregated long skills text
        - Approach A: Traditional/Sparse matching model (TF-IDF, BM25, QLM)
            - Bag-of-words (BoWs) representation + TF-IDF model
            - BoWs representation + BM25 model
        - Approach B: Dense matching model (BERT, RoBERTa, GPT)
            - Contextual representation (Sentence Transformers) + Cosine similarity

##### **Transform `skill_aliases` column to a list of strings**

In [30]:
validation_corpus_elements["skill_aliases"] = validation_corpus_elements["skill_aliases"].apply(lambda x: ast.literal_eval(x))

In [31]:
validation_corpus_elements.head()

,c_id,esco_uri,skill_aliases
0,dev_cb_sk_1,http://data.europa.eu/esco/skill/1c460d2d-90c6...,"[pricing plans, price strategies, pricing tact..."
1,dev_cb_sk_2,http://data.europa.eu/esco/skill/301a6581-e983...,"[putting out fires, put out fires, extinguish ..."
2,dev_cb_sk_3,http://data.europa.eu/esco/skill/a4881e54-6055...,"[online assessment, analysis of web strategy, ..."
3,dev_cb_sk_4,http://data.europa.eu/esco/skill/efda73b4-5212...,"[keeping up with trends, keep pace with trends..."
4,dev_cb_sk_5,http://data.europa.eu/esco/skill/22a173f5-868c...,"[prepare tax returns form, complete tax return..."


##### **Aggregate the skill aliases into one long text**

In [32]:
def aggregate_list_of_terms(terms_list):
    '''
        aggregate the list of terms to a long text (string)
    '''
    return " ".join(terms_list)

In [33]:
validation_corpus_elements["skill_document"] = validation_corpus_elements["skill_aliases"].apply(lambda x: aggregate_list_of_terms(x))

In [34]:
validation_corpus_elements.head()

,c_id,esco_uri,skill_aliases,skill_document
0,dev_cb_sk_1,http://data.europa.eu/esco/skill/1c460d2d-90c6...,"[pricing plans, price strategies, pricing tact...",pricing plans price strategies pricing tactics...
1,dev_cb_sk_2,http://data.europa.eu/esco/skill/301a6581-e983...,"[putting out fires, put out fires, extinguish ...",putting out fires put out fires extinguish fir...
2,dev_cb_sk_3,http://data.europa.eu/esco/skill/a4881e54-6055...,"[online assessment, analysis of web strategy, ...",online assessment analysis of web strategy web...
3,dev_cb_sk_4,http://data.europa.eu/esco/skill/efda73b4-5212...,"[keeping up with trends, keep pace with trends...",keeping up with trends keep pace with trends f...
4,dev_cb_sk_5,http://data.europa.eu/esco/skill/22a173f5-868c...,"[prepare tax returns form, complete tax return...",prepare tax returns form complete tax returns ...


In [35]:
validation_corpus_elements.skill_document[0]

'pricing plans price strategies pricing tactics pricing strategies pricing strategy'

#### Generating a mapping (dictionary) for queries

In [44]:
'''
    build a dictionary/mapping between the query_id and text from the query pandas dataframe
'''
validation_queries_ids = validation_queries.q_id.to_list()
validation_queries_texts = validation_queries.jobtitle.to_list()

validation_queries_map = dict(zip(validation_queries_ids, validation_queries_texts))

In [45]:
list(validation_queries_map.items())[0]

('dev_qb_jt_1', 'corporate governance analyst')

In [46]:
validation_corpus_elements.head()

,c_id,esco_uri,skill_aliases,skill_document
0,dev_cb_sk_1,http://data.europa.eu/esco/skill/1c460d2d-90c6...,"[pricing plans, price strategies, pricing tact...",pricing plans price strategies pricing tactics...
1,dev_cb_sk_2,http://data.europa.eu/esco/skill/301a6581-e983...,"[putting out fires, put out fires, extinguish ...",putting out fires put out fires extinguish fir...
2,dev_cb_sk_3,http://data.europa.eu/esco/skill/a4881e54-6055...,"[online assessment, analysis of web strategy, ...",online assessment analysis of web strategy web...
3,dev_cb_sk_4,http://data.europa.eu/esco/skill/efda73b4-5212...,"[keeping up with trends, keep pace with trends...",keeping up with trends keep pace with trends f...
4,dev_cb_sk_5,http://data.europa.eu/esco/skill/22a173f5-868c...,"[prepare tax returns form, complete tax return...",prepare tax returns form complete tax returns ...


#### Generaring a mapping for Documents (Corpus elements)

In [48]:
'''
    build a dictionary/mapping between the corpus_id and document from the pandas dataframe
'''
validation_documents_ids = validation_corpus_elements.c_id.to_list()
validation_documents_esco_uri = validation_corpus_elements.esco_uri.to_list()
validation_documents_texts = validation_corpus_elements.skill_document.to_list()

validation_documents_map = dict(zip(validation_documents_ids, validation_documents_texts))

In [50]:
list(validation_documents_map.items())[0]

('dev_cb_sk_1',
 'pricing plans price strategies pricing tactics pricing strategies pricing strategy')

#### **Loading Contextual Embedding model**

In [51]:
model = SentenceTransformer("all-MiniLM-L6-v2", token=False)

In [52]:
#tracker = EmissionsTracker()
#tracker.start_task("all-MiniLM-L6-v2")

In [53]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

#### **Embedding Queries and Documents**
- Dense representation using SentenceTransformers

In [54]:
validation_query_embeddings = model.encode(validation_queries_texts, convert_to_tensor=True)

In [55]:
validation_query_embeddings.shape

torch.Size([304, 384])

In [56]:
validation_documents_embeddings = model.encode(validation_documents_texts, convert_to_tensor=True)

In [58]:
validation_documents_embeddings.shape 

torch.Size([1439, 384])

##### Cosine Similarity between Embedding Representation

In [60]:
validation_similarities_query_documents = util.cos_sim(validation_query_embeddings, validation_documents_embeddings).cpu().numpy()

In [61]:
validation_similarities_query_documents.shape

(304, 1439)

In [62]:
validation_similarities_query_documents

array([[ 0.212381  ,  0.04687777,  0.17210871, ...,  0.05702206,
         0.1703934 ,  0.0501546 ],
       [ 0.23119703,  0.05267682,  0.1957598 , ...,  0.03260248,
         0.12111831, -0.00484402],
       [ 0.31889713,  0.12826607,  0.1675432 , ...,  0.17722419,
         0.25939736,  0.00225326],
       ...,
       [ 0.05365921,  0.03072052,  0.13522762, ...,  0.11323761,
         0.07504336, -0.01391865],
       [ 0.03805338,  0.03404774,  0.13891795, ...,  0.04707766,
         0.14684533,  0.14173211],
       [ 0.05216486, -0.03773117,  0.19666046, ...,  0.03985806,
         0.23905906,  0.07785819]], dtype=float32)

## Evaluation Preparation
- To compute the evaluation metrics, we will generate a submission file using the query-documents similarity metric.
- The submission file must follow the ``TREC Run File format'', including headers in the output file.
- This means that the file should have 6 space-spearated columns per line, with following information:
    - q_id: query ID.
    - Q0: a constant identifier, usually "Q0".
    - doc_id: id of the retrieved corpus elements (skill_id).
    - rank: position of the document in the ranking.
    - score: relevance score assigned by the model.
    - tag: experiment name

### Preparing Submission File

In [64]:
def get_ranked_result_list(queries_ids, queries_map, documents_ids, documents_map, similarities_query_documents, documents_texts, model_name):
    results = []
    results_name = []
    
    for q_idx, q_id in enumerate(queries_ids):
        sorted_indices = np.argsort(-similarities_query_documents[q_idx])
        used_doc_ids = set()
        rank_counter = 0
        for c_idx in sorted_indices:  # Consider the full list.
            doc_id = documents_ids[c_idx]
            # If doc_id was already processed, go to the next one.
            if doc_id in used_doc_ids:
                continue
            used_doc_ids.add(doc_id)
            rank_counter += 1
    
            query_name = queries_map[q_id]
            doc_name = documents_texts[c_idx]
            score = similarities_query_documents[q_idx, c_idx]
    
            results.append(f"{q_id} Q0 {doc_id} {rank_counter} {score:.4f} {model_name}")
            results_name.append(f"{query_name} Q0 {doc_name} {rank_counter} {score:.4f} {model_name}")

    return results, results_name

In [65]:
results_list, results_name = get_ranked_result_list(validation_queries_ids, validation_queries_map,
                                                    validation_documents_ids, validation_documents_map,
                                                    validation_similarities_query_documents, validation_documents_texts,
                                                   "Adhoc_ST_Cosine")

In [66]:
results_path

'results'

In [67]:
result_adhoc_STC_taskB = os.path.join(results_path, "validation_adhoc_STC_taskB.trec")
with open(result_adhoc_STC_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_list))

### Evaluation Scripts

- For the evaluation, we will use the official [TalentCLEF evaluation script](https://github.com/TalentCLEF/talentclef25_evaluation_script), which uses the Ranx library under the hood.

- First, clone the repo and install the requirements file:

In [70]:
# run this cell just once for the first time
!git clone https://github.com/TalentCLEF/talentclef25_evaluation_script.git
!pip install -r talentclef25_evaluation_script/requirements.txt

Cloning into 'talentclef25_evaluation_script'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 27 (delta 10), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 10.10 KiB | 1.12 MiB/s, done.
Resolving deltas: 100% (10/10), done.
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 55.7 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 41.4 MB/s eta 0:00:

Then, select the Qrels file and the Run file to perform the evaluation.

In [72]:
qrels_file = os.path.join(validation_data_path, "qrels.tsv")

run_validation_adhoc_STC_file = os.path.join(results_path, "validation_adhoc_STC_taskB.trec")

In [73]:
print ("Evaluation")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_validation_adhoc_STC_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

Evaluation
Received parameters:
  qrels: taskB/validation/qrels.tsv
  run: results/validation_adhoc_STC_taskB.trec
Loading qrels...
Loading run...
Running evaluation...

=== Evaluation Results ===
map: 0.1755
mrr: 0.6756
ndcg: 0.6564
precision@5: 0.4336
precision@10: 0.3766
precision@100: 0.1859



In [75]:
# Alternative approach for computing evaluation metrics

!python talentclef25_evaluation_script/talentclef_evaluate.py --qrels taskB/validation/qrels.tsv --run results/validation_adhoc_STC_taskB.trec

Received parameters:
  qrels: taskB/validation/qrels.tsv
  run: results/validation_adhoc_STC_taskB.trec
Loading qrels...
Loading run...
Running evaluation...

=== Evaluation Results ===
map: 0.1755
mrr: 0.6756
ndcg: 0.6564
precision@5: 0.4336
precision@10: 0.3766
precision@100: 0.1859
